# Prompt Engineering to Agentic RAG 🤖🧠

## LangChain RAG Agent with AG News

This notebook builds a Retrieval-Augmented Generation agent with LangChain and the public **AG News** dataset from Hugging Face.

It demonstrates:

- prompt engineering;
- context optimization based on token count;
- query rewriting;
- query expansion;
- [step-back prompting](https://www.langchain.com/blog/query-transformations#step-back-prompting);
- HyDE;
- baseline versus transformed retrieval.

> AG News is historical. Treat it as a news archive, not a live news service.

## Learning objectives

1. Load and index a public Hugging Face dataset.
2. Build a LangChain retrieval tool and RAG agent.
3. Optimize context with truncation, extractive compression, and conditional summarization.
4. Apply query rewriting, expansion, step-back prompting, and HyDE.
5. Merge and compare retrieval results.
6. Compare baseline and transformed RAG pipelines.

## 1. Install dependencies

In [111]:
%pip install -U datasets langchain langchain-openai langchain-community langchain-text-splitters python-dotenv colorama tiktoken pandas

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


--- Logging error ---
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_internal/utils/logging.py", line 177, in emit
    self.console.print(renderable, overflow="ignore", crop=False, style=style)
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_vendor/rich/console.py", line 1673, in print
    extend(render(renderable, render_options))
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_vendor/rich/console.py", line 1305, in render
    for render_output in iter_render:
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_internal/utils/logging.py", line 134, in __rich_console__
    for line in lines:
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_vendor/rich/segment.py", line 249, in split_lines
    for segment in segments:
  File "/Library/

## 2. Environment setup

In [112]:
import os
from getpass import getpass

from colorama import Fore, Style
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
else:
    print(f"{Fore.GREEN}OpenAI API key loaded.{Style.RESET_ALL}")

OpenAI API key loaded.


## 3. Initialize the model

In [113]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-4.1-mini",
    temperature=0,
    timeout=30,
    max_retries=3,
)

print(f"{Fore.CYAN}Model initialized.{Style.RESET_ALL}")

Model initialized.


## 4. Define prompts

In [114]:
WEAK_PROMPT = """You are a news assistant.
Use the available search tool and answer the user's question."""

GROUNDED_PROMPT = """You are a grounded news-archive assistant.

Use the available search tool before answering questions about the archive.

Rules:
- Use only retrieved evidence.
- Cite factual claims with [Source N].
- Never present historical archive data as current.
- If evidence is insufficient, say so.
- Begin with a direct answer.
- Add up to three concise cited bullets.
- Finish with one limitation.
- Use no more than 180 words.
"""

BEGINNER_FRIENDLY_PROMPT = """You are a teacher explaining historical news to beginners.

Use the available search tool and only retrieved evidence.
Explain difficult terms, use at most three bullets, include one example,
cite factual claims with [Source N], and end with 'In simple terms, ...'.
Write 140 to 180 words.
"""

EXECUTIVE_PROMPT = """You are preparing a briefing for a business leader.

Use the available search tool and only retrieved evidence.

Return exactly:
## Headline
## Key Signals
## Business Relevance
## Data Limitation

Use direct language and no more than 140 words.
"""

CLARIFYING_PROMPT = """You are a grounded news-archive assistant.

If the request is ambiguous, ask one concise clarification question and do not search yet.
Otherwise use the available search tool, cite [Source N], and state when evidence is insufficient.
"""

ANALYTICAL_PROMPT = """You are an analytical research assistant.

Use the available search tool and only retrieved evidence.

Return exactly:
## Main Finding
## Evidence
## Interpretation
## Limitation

Use neutral language and no more than 220 words.
"""

## 5. Load the AG News dataset

In [115]:
from datasets import load_dataset

dataset = load_dataset(
    "fancyzhx/ag_news",
    split="train[:500]",
)

print(dataset)
print(dataset[0])

Dataset({
    features: ['text', 'label'],
    num_rows: 500
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


## 6. Convert rows into LangChain documents

In [116]:
from langchain_core.documents import Document

LABELS = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Science and Technology",
}

documents = []

for index, row in enumerate(dataset):
    label = int(row["label"])
    documents.append(
        Document(
            page_content=row["text"].strip(),
            metadata={
                "source": f"ag_news_{index}",
                "category": LABELS[label],
                "dataset": "AG News",
                "record_id": index,
            },
        )
    )

print(f"Documents created: {len(documents)}")
print(documents[0])

Documents created: 500
page_content='Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.' metadata={'source': 'ag_news_0', 'category': 'Business', 'dataset': 'AG News', 'record_id': 0}


## 7. Split, embed, and index

In [117]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
)

vector_store = InMemoryVectorStore(
    embedding=embeddings,
)

vector_store.add_documents(chunks)

baseline_retriever = vector_store.as_retriever(
    search_kwargs={"k": 5},
)

broad_retriever = vector_store.as_retriever(
    search_kwargs={"k": 12},
)

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")

Documents: 500
Chunks: 524


## 8. Formatting and token utilities

In [118]:
import tiktoken

TOKENIZER = tiktoken.encoding_for_model("gpt-4.1-mini")


def count_tokens(text: str) -> int:
    return len(TOKENIZER.encode(text))


def document_tokens(doc: Document) -> int:
    return count_tokens(doc.page_content)


def format_documents(docs: list[Document]) -> str:
    if not docs:
        return "No relevant articles were retrieved."

    parts = []

    for position, doc in enumerate(docs, start=1):
        parts.append(
            f"[Source {position}]\n"
            f"Record: {doc.metadata.get('source', 'unknown')}\n"
            f"Category: {doc.metadata.get('category', 'unknown')}\n"
            f"Article: {doc.page_content}"
        )

    return "\n\n".join(parts)


def inspect_token_usage(
    docs: list[Document],
    label: str = "Context",
) -> dict:
    counts = [document_tokens(doc) for doc in docs]

    stats = {
        "label": label,
        "documents": len(docs),
        "total_tokens": sum(counts),
        "average_tokens": round(sum(counts) / len(counts), 1) if counts else 0,
        "largest_document": max(counts, default=0),
    }

    print(stats)
    return stats

## 9. Context optimization

The optimizer runs automatically from token counts:

1. Keep full context when it fits.
2. Compress relevant sentences when it is too large.
3. Truncate to the token budget.
4. Summarize only if the result remains above the preferred threshold.

In [119]:
import re
from math import sqrt

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


def truncate_documents_by_tokens(
    docs: list[Document],
    max_tokens: int = 900,
) -> tuple[list[Document], dict]:
    selected = []
    excluded = []
    used_tokens = 0

    for doc in docs:
        tokens = document_tokens(doc)

        if used_tokens + tokens <= max_tokens:
            selected.append(doc)
            used_tokens += tokens
        else:
            excluded.append(doc)

    return selected, {
        "budget": max_tokens,
        "used_tokens": used_tokens,
        "selected_documents": len(selected),
        "excluded_documents": len(excluded),
    }


def cosine_similarity(
    vector_a: list[float],
    vector_b: list[float],
) -> float:
    numerator = sum(a * b for a, b in zip(vector_a, vector_b))
    norm_a = sqrt(sum(a * a for a in vector_a))
    norm_b = sqrt(sum(b * b for b in vector_b))

    if not norm_a or not norm_b:
        return 0.0

    return numerator / (norm_a * norm_b)


def split_sentences(text: str) -> list[str]:
    return [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+", text)
        if sentence.strip()
    ]


def compress_documents(
    query: str,
    docs: list[Document],
    threshold: float = 0.20,
    max_sentences_per_doc: int = 2,
) -> list[Document]:
    if not docs:
        return []

    query_vector = embeddings.embed_query(query)
    document_sentences = [
        split_sentences(doc.page_content)
        for doc in docs
    ]
    all_sentences = [
        sentence
        for sentences in document_sentences
        for sentence in sentences
    ]

    if not all_sentences:
        return []

    all_vectors = embeddings.embed_documents(all_sentences)

    output = []
    vector_index = 0

    for doc, sentences in zip(docs, document_sentences):
        sentence_count = len(sentences)
        vectors = all_vectors[
            vector_index: vector_index + sentence_count
        ]
        vector_index += sentence_count

        scored_sentences = sorted(
            [
                (
                    cosine_similarity(query_vector, vector),
                    sentence,
                )
                for sentence, vector in zip(sentences, vectors)
            ],
            key=lambda item: item[0],
            reverse=True,
        )

        kept = [
            sentence
            for score, sentence in scored_sentences[:max_sentences_per_doc]
            if score >= threshold
        ]

        if kept:
            output.append(
                Document(
                    page_content=" ".join(kept),
                    metadata={
                        **doc.metadata,
                        "compression": "extractive",
                    },
                )
            )

    return output


SUMMARY_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Compress retrieved news evidence for a downstream RAG answer.

Preserve:
- facts needed to answer the question;
- source labels exactly as [Source N];
- separate events;
- uncertainty and limitations.

Do not add outside information.
Return no more than {target_words} words.""",
        ),
        (
            "human",
            """Question:
{question}

Retrieved evidence:
{context}

Compressed evidence:""",
        ),
    ]
)

summary_chain = SUMMARY_PROMPT | model | StrOutputParser()


def summarize_context(
    question: str,
    docs: list[Document],
    target_words: int = 180,
) -> str:
    return summary_chain.invoke(
        {
            "question": question,
            "context": format_documents(docs),
            "target_words": target_words,
        }
    )


def optimize_context(
    query: str,
    docs: list[Document],
    max_context_tokens: int = 900,
    summarize_threshold: int = 700,
) -> tuple[str, dict]:
    before_context = format_documents(docs)
    before_tokens = count_tokens(before_context)

    details = {
        "action": "full_context",
        "documents_before": len(docs),
    }

    if before_tokens <= max_context_tokens:
        context = before_context

    else:
        compressed_docs = compress_documents(
            query=query,
            docs=docs,
        )

        if not compressed_docs:
            compressed_docs = docs
            details["compression_fallback"] = True

        details["documents_after_compression"] = len(compressed_docs)

        selected_docs, truncation_report = truncate_documents_by_tokens(
            docs=compressed_docs,
            max_tokens=max_context_tokens,
        )

        context = format_documents(selected_docs)
        details.update(truncation_report)
        details["action"] = "compress_then_truncate"

        if selected_docs and count_tokens(context) > summarize_threshold:
            context = summarize_context(
                question=query,
                docs=selected_docs,
                target_words=180,
            )
            details["action"] = "compress_truncate_then_summarize"

    after_tokens = count_tokens(context)

    return context, {
        "before_tokens": before_tokens,
        "after_tokens": after_tokens,
        "token_reduction": before_tokens - after_tokens,
        **details,
    }

### Context optimization demo

In [120]:
optimization_query = "technology companies and business risks"
optimization_docs = broad_retriever.invoke(optimization_query)

optimized_context, optimization_report = optimize_context(
    query=optimization_query,
    docs=optimization_docs,
)

print(optimization_report)
print("\nOptimized context:\n")
print(optimized_context[:1500])

{'before_tokens': 837, 'after_tokens': 837, 'token_reduction': 0, 'action': 'full_context', 'documents_before': 12}

Optimized context:

[Source 1]
Record: ag_news_42
Category: Business
Article: Technology company sues five ex-employees  A Marlborough-based technology company is suing five former employees, including three senior managers, for allegedly conspiring against their employer while working on opening a competing business.

[Source 2]
Record: ag_news_322
Category: Science and Technology
Article: Tech market indeed soft, but HP woes one of a kind HP and Cisco spooked investors with recent comments about sales, but analysts say the overall tech market is still inching along.

[Source 3]
Record: ag_news_310
Category: Science and Technology
Article: If this is a tech rebound, pinch me KnowledgeWharton goes inside the IT conundrum to examine why capital spending--and corporate confidence--remains low.

[Source 4]
Record: ag_news_309
Category: Science and Technology
Article: Gettin

# Query Transformation

The raw user question may be vague or use different vocabulary from the archive.

This section adds four complementary techniques:

1. **Query rewriting** — creates one clearer retrieval query.
2. **Query expansion** — generates alternative phrasings.
3. **Step-back prompting** — retrieves broader conceptual context.
4. **HyDE** — retrieves using an embedding of a hypothetical answer.

## 10. Query rewriting

In [121]:
REWRITE_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Rewrite the user's question as one clear, keyword-rich search query
for a historical news archive.

Do not answer the question.
Return only the rewritten query.""",
        ),
        ("human", "{question}"),
    ]
)

rewrite_chain = REWRITE_PROMPT | model | StrOutputParser()


def rewrite_query(question: str) -> str:
    return rewrite_chain.invoke(
        {"question": question}
    ).strip()


example_question = "what's going on with tech stocks lately"
rewritten = rewrite_query(example_question)

print(f"Original: {example_question}")
print(f"Rewritten: {rewritten}")

Original: what's going on with tech stocks lately
Rewritten: recent trends and developments in technology stocks 2024


## 11. Query expansion

In [122]:
EXPANSION_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Generate exactly {n} retrieval queries that explore different
phrasings, synonyms, or angles of the user's question.

Return one query per line.
Do not use numbering or add explanations.""",
        ),
        ("human", "{question}"),
    ]
)

expansion_chain = EXPANSION_PROMPT | model | StrOutputParser()


def expand_query(
    question: str,
    n: int = 3,
) -> list[str]:
    raw = expansion_chain.invoke(
        {
            "question": question,
            "n": n,
        }
    )

    queries = [
        line.strip("-• ").strip()
        for line in raw.splitlines()
        if line.strip()
    ]

    return queries[:n]


def retrieve_with_expansion(
    question: str,
    n: int = 3,
    k: int = 5,
) -> list[Document]:
    queries = [question, *expand_query(question, n=n)]

    merged = []
    seen_sources = set()

    for query in queries:
        for doc in baseline_retriever.invoke(query):
            source = doc.metadata.get("source", doc.id)

            if source not in seen_sources:
                seen_sources.add(source)
                merged.append(doc)

    return merged[:k]


example_question = "technology companies and business risks"
expanded_queries = expand_query(example_question)

print(expanded_queries)
inspect_token_usage(
    retrieve_with_expansion(example_question),
    "Expansion retrieval",
)

['business risks faced by technology companies', 'common challenges and risks in tech industry businesses', 'how technology companies manage business risks']
{'label': 'Expansion retrieval', 'documents': 5, 'total_tokens': 215, 'average_tokens': 43.0, 'largest_document': 72}


{'label': 'Expansion retrieval',
 'documents': 5,
 'total_tokens': 215,
 'average_tokens': 43.0,
 'largest_document': 72}

## 12. Step-back prompting

In [123]:
STEP_BACK_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Rewrite the user's question as one broader, higher-level question
that captures the underlying concepts.

Do not answer the question.
Return only the broader question.""",
        ),
        ("human", "{question}"),
    ]
)

step_back_chain = STEP_BACK_PROMPT | model | StrOutputParser()


def step_back_query(question: str) -> str:
    return step_back_chain.invoke(
        {"question": question}
    ).strip()


def retrieve_with_step_back(
    question: str,
    k: int = 5,
) -> tuple[list[Document], str]:
    broader_question = step_back_query(question)
    documents = baseline_retriever.invoke(broader_question)[:k]
    return documents, broader_question


example_question = "What business risks do technology companies face?"
step_back_docs, broader_question = retrieve_with_step_back(
    example_question
)

print(f"Original: {Fore.CYAN}{example_question}{Fore.RESET}")
print(f"Step-back: {Fore.MAGENTA}{broader_question}{Fore.RESET}")
inspect_token_usage(step_back_docs, "Step-back retrieval")

Original: What business risks do technology companies face?
Step-back: What are the various types of risks that technology companies encounter in their business operations?
{'label': 'Step-back retrieval', 'documents': 5, 'total_tokens': 209, 'average_tokens': 41.8, 'largest_document': 66}


{'label': 'Step-back retrieval',
 'documents': 5,
 'total_tokens': 209,
 'average_tokens': 41.8,
 'largest_document': 66}

## 13. HyDE

In [124]:
HYDE_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Write a short hypothetical historical-news passage that could
answer the user's question.

Do not say it is hypothetical.
Use two or three sentences.""",
        ),
        ("human", "{question}"),
    ]
)

hyde_chain = HYDE_PROMPT | model | StrOutputParser()


def hyde_retrieve(
    question: str,
    k: int = 5,
) -> tuple[list[Document], str]:
    hypothetical_document = hyde_chain.invoke(
        {"question": question}
    ).strip()

    hyde_vector = embeddings.embed_query(
        hypothetical_document
    )

    documents = vector_store.similarity_search_by_vector(
        hyde_vector,
        k=k,
    )

    return documents, hypothetical_document


example_question = "technology companies and business risks"
hyde_docs, hypothetical_document = hyde_retrieve(
    example_question
)

print("Hypothetical document:")
print(hypothetical_document)
inspect_token_usage(hyde_docs, "HyDE retrieval")

Hypothetical document:
In recent years, technology companies have increasingly faced a variety of business risks, including cybersecurity threats, regulatory challenges, and rapid market changes. For instance, in 2023, several major tech firms reported significant financial impacts due to data breaches and evolving privacy laws, highlighting the critical need for robust risk management strategies in the industry.
{'label': 'HyDE retrieval', 'documents': 5, 'total_tokens': 227, 'average_tokens': 45.4, 'largest_document': 60}


{'label': 'HyDE retrieval',
 'documents': 5,
 'total_tokens': 227,
 'average_tokens': 45.4,
 'largest_document': 60}

## 14. Unified transformed retrieval

In [125]:
def document_identity(doc: Document) -> str:
    return str(
        doc.metadata.get("source")
        or doc.metadata.get("record_id")
        or doc.id
    )


def merge_unique_documents(
    *document_lists: list[Document],
) -> list[Document]:
    merged = []
    seen = set()

    for documents_list in document_lists:
        for doc in documents_list:
            identity = document_identity(doc)

            if identity not in seen:
                seen.add(identity)
                merged.append(doc)

    return merged


def retrieve_with_all_transformations(
    question: str,
    k: int = 8,
) -> tuple[list[Document], dict]:
    rewritten_query = rewrite_query(question)
    expanded_queries = expand_query(question)

    baseline_docs = baseline_retriever.invoke(question)
    rewritten_docs = baseline_retriever.invoke(rewritten_query)
    expanded_docs = retrieve_with_expansion(
        question=question,
        k=k,
    )
    step_back_docs, broader_question = retrieve_with_step_back(
        question=question,
        k=k,
    )
    hyde_docs, hypothetical_document = hyde_retrieve(
        question=question,
        k=k,
    )

    merged_docs = merge_unique_documents(
        baseline_docs,
        rewritten_docs,
        expanded_docs,
        step_back_docs,
        hyde_docs,
    )[:k]

    details = {
        "rewritten_query": rewritten_query,
        "expanded_queries": expanded_queries,
        "step_back_query": broader_question,
        "hypothetical_document": hypothetical_document,
        "baseline_documents": len(baseline_docs),
        "rewrite_documents": len(rewritten_docs),
        "expansion_documents": len(expanded_docs),
        "step_back_documents": len(step_back_docs),
        "hyde_documents": len(hyde_docs),
        "documents_retrieved": len(merged_docs),
        "sources": [
            document_identity(doc)
            for doc in merged_docs
        ],
    }

    return merged_docs, details

### Compare individual retrieval techniques

In [126]:
import pandas as pd


def compare_retrieval_techniques(
    question: str,
    k: int = 5,
) -> pd.DataFrame:
    baseline_docs = baseline_retriever.invoke(question)
    rewritten_docs = baseline_retriever.invoke(
        rewrite_query(question)
    )
    expanded_docs = retrieve_with_expansion(
        question=question,
        k=k,
    )
    step_back_docs, _ = retrieve_with_step_back(
        question=question,
        k=k,
    )
    hyde_docs, _ = hyde_retrieve(
        question=question,
        k=k,
    )
    combined_docs, _ = retrieve_with_all_transformations(
        question=question,
        k=k,
    )

    baseline_sources = {
        document_identity(doc)
        for doc in baseline_docs
    }

    results = {
        "baseline": baseline_docs,
        "rewrite": rewritten_docs,
        "expansion": expanded_docs,
        "step_back": step_back_docs,
        "hyde": hyde_docs,
        "combined": combined_docs,
    }

    rows = []

    for technique, docs in results.items():
        sources = {
            document_identity(doc)
            for doc in docs
        }

        rows.append(
            {
                "technique": technique,
                "documents_retrieved": len(docs),
                "unique_sources": len(sources),
                "new_vs_baseline": len(
                    sources - baseline_sources
                ),
            }
        )

    return pd.DataFrame(rows)


comparison_question = "technology companies and business risks"
compare_retrieval_techniques(comparison_question)

,technique,documents_retrieved,unique_sources,new_vs_baseline
0,baseline,5,5,0
1,rewrite,5,5,1
2,expansion,5,5,0
3,step_back,5,5,1
4,hyde,5,5,2
5,combined,5,5,0


## 15. Create retrieval tools

In [127]:
from langchain.tools import tool


@tool
def search_news_archive(query: str) -> str:
    """Search the historical AG News archive without context optimization."""

    docs = baseline_retriever.invoke(query)

    if not docs:
        return (
            "STATUS: NO_RESULTS\n"
            "No relevant articles were found."
        )

    context = format_documents(docs)

    return (
        "STATUS: RESULTS_FOUND\n"
        f"CONTEXT_TOKENS: {count_tokens(context)}\n\n"
        f"{context}"
    )


@tool
def search_optimized_news_archive(query: str) -> str:
    """Search AG News and optimize context automatically from token count."""

    docs = broad_retriever.invoke(query)

    if not docs:
        return (
            "STATUS: NO_RESULTS\n"
            "No relevant articles were found."
        )

    context, report = optimize_context(
        query=query,
        docs=docs,
    )

    return (
        "STATUS: RESULTS_FOUND\n"
        f"OPTIMIZATION_ACTION: {report['action']}\n"
        f"TOKENS_BEFORE: {report['before_tokens']}\n"
        f"TOKENS_AFTER: {report['after_tokens']}\n"
        f"TOKEN_REDUCTION: {report['token_reduction']}\n\n"
        f"{context}"
    )


@tool
def search_news_archive_transformed(query: str) -> str:
    """Search AG News with rewriting, expansion, step-back prompting, and HyDE."""

    docs, transformation_report = retrieve_with_all_transformations(
        question=query
    )

    if not docs:
        return (
            "STATUS: NO_RESULTS\n"
            "No relevant articles were found."
        )

    context, optimization_report = optimize_context(
        query=query,
        docs=docs,
    )

    return (
        "STATUS: RESULTS_FOUND\n"
        f"REWRITTEN_QUERY: {transformation_report['rewritten_query']}\n"
        f"EXPANDED_QUERIES: {transformation_report['expanded_queries']}\n"
        f"STEP_BACK_QUERY: {transformation_report['step_back_query']}\n"
        f"DOCUMENTS_RETRIEVED: {transformation_report['documents_retrieved']}\n"
        f"OPTIMIZATION_ACTION: {optimization_report['action']}\n"
        f"TOKENS_BEFORE: {optimization_report['before_tokens']}\n"
        f"TOKENS_AFTER: {optimization_report['after_tokens']}\n\n"
        f"{context}"
    )

## 16. Create agents

In [128]:
from langchain.agents import create_agent

baseline_agent = create_agent(
    model=model,
    tools=[search_news_archive],
    system_prompt=GROUNDED_PROMPT,
)

optimized_agent = create_agent(
    model=model,
    tools=[search_optimized_news_archive],
    system_prompt=GROUNDED_PROMPT,
)

transformed_agent = create_agent(
    model=model,
    tools=[search_news_archive_transformed],
    system_prompt=GROUNDED_PROMPT,
)

prompt_agents = {
    "Weak": create_agent(
        model=model,
        tools=[search_news_archive],
        system_prompt=WEAK_PROMPT,
    ),
    "Grounded": baseline_agent,
    "Beginner": create_agent(
        model=model,
        tools=[search_news_archive],
        system_prompt=BEGINNER_FRIENDLY_PROMPT,
    ),
    "Executive": create_agent(
        model=model,
        tools=[search_news_archive],
        system_prompt=EXECUTIVE_PROMPT,
    ),
    "Clarifying": create_agent(
        model=model,
        tools=[search_news_archive],
        system_prompt=CLARIFYING_PROMPT,
    ),
    "Analytical": create_agent(
        model=model,
        tools=[search_news_archive],
        system_prompt=ANALYTICAL_PROMPT,
    ),
}

print("Agents created.")

Agents created.


## 17. Invoke agents and print logs

In [129]:
import time


def ask_agent(
    agent,
    question: str,
    label: str,
) -> dict:
    print("\n" + "=" * 80)
    print(f"{Fore.CYAN}{label}{Style.RESET_ALL}")
    print("=" * 80)
    print(f"{Fore.YELLOW}[Question]{Style.RESET_ALL}")
    print(question)

    started = time.perf_counter()

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        }
    )

    latency = time.perf_counter() - started
    answer = result["messages"][-1].content

    output = {
        "pipeline": label,
        "latency_seconds": round(latency, 3),
        "answer_tokens": count_tokens(answer),
        "answer": answer,
    }

    print(f"\n{Fore.YELLOW}[Latency]{Style.RESET_ALL}")
    print(f"{output['latency_seconds']} seconds")

    print(f"\n{Fore.YELLOW}[Answer tokens]{Style.RESET_ALL}")
    print(output["answer_tokens"])

    print(f"\n{Fore.GREEN}[Answer]{Style.RESET_ALL}")
    print(answer)

    return output

### Baseline, optimized, and transformed comparison

In [130]:
question = "What business risks do technology companies face?"

comparison_results = [
    ask_agent(
        agent=baseline_agent,
        question=question,
        label="Baseline RAG",
    ),
    ask_agent(
        agent=optimized_agent,
        question=question,
        label="Context-Optimized RAG",
    ),
    ask_agent(
        agent=transformed_agent,
        question=question,
        label="Transformed RAG with Step-Back",
    ),
]

comparison_df = pd.DataFrame(
    [
        {
            "pipeline": result["pipeline"],
            "latency_seconds": result["latency_seconds"],
            "answer_tokens": result["answer_tokens"],
        }
        for result in comparison_results
    ]
)

# comparison_df


Baseline RAG
[Question]
What business risks do technology companies face?

[Latency]
3.424 seconds

[Answer tokens]
186

[Answer]
Technology companies face several business risks including internal threats such as employee conspiracies to start competing businesses, which can lead to legal disputes and loss of intellectual property [Source 1]. They also encounter market risks like fluctuating corporate confidence and capital spending, which can impact growth and investment [Source 2]. Additionally, the tech market can be soft, with companies like HP and Cisco experiencing sales challenges that spook investors, indicating sector-specific vulnerabilities [Source 3]. Security risks are significant as well, with network security often becoming a costly and complex issue due to fragmented approaches, potentially leading to financial and operational setbacks [Source 5].

- Internal employee conflicts and competition risks [Source 1].
- Market volatility affecting investment and sales [Sourc

### Prompt-variant comparison

In [131]:
prompt_question = "What do the articles say about technology companies?"

for name, agent in prompt_agents.items():
    ask_agent(
        agent=agent,
        question=prompt_question,
        label=name,
    )


Weak
[Question]
What do the articles say about technology companies?

[Latency]
4.25 seconds

[Answer tokens]
200

[Answer]
The articles about technology companies highlight several points:

1. The tech market is somewhat soft, with specific companies like HP and Cisco causing investor concern due to sales issues, but overall the tech market is still progressing slowly (Source 1).

2. There is a perspective from McKinsey that the post-boom tech industry can still be very profitable by focusing on operations in slower-growth industries (Source 2).

3. A technology company in Marlborough is involved in a lawsuit against five former employees, including senior managers, who allegedly conspired to start a competing business (Source 3).

4. Despite some signs of a tech rebound, capital spending and corporate confidence in IT remain low, indicating ongoing challenges in the sector (Source 4).

5. Dell is performing well, showing positive results in contrast to the broader tech industry, whi

## 18. Ambiguity and insufficient-context tests

In [132]:
ask_agent(
    agent=prompt_agents["Clarifying"],
    question="What happened with Apple?",
    label="Clarification Test",
)

ask_agent(
    agent=baseline_agent,
    question="What did Apple announce yesterday?",
    label="Insufficient Context Test",
)


Clarification Test
[Question]
What happened with Apple?

[Latency]
3.295 seconds

[Answer tokens]
149

[Answer]
Recent news about Apple includes several developments:

1. Apple has begun shipping Motion, a high-performance, real-time motion graphics design software integrated with Final Cut Pro HD and DVD Studio Pro 3, priced at $299. [Source 1]

2. A Norwegian hacker known for cracking DVD encryption has reportedly broken the locks on Apple's wireless music streaming technology. [Source 2, Source 4]

3. Apple is set to open its second retail store in Japan later this month, located in Osaka. [Source 3]

4. In the competitive market, a Dutch music retailer has launched a download service ahead of Apple in Europe. [Source 5]

If you want information on a specific aspect of Apple, please let me know!

Insufficient Context Test
[Question]
What did Apple announce yesterday?

[Latency]
3.138 seconds

[Answer tokens]
129

[Answer]
Apple announced the shipping of Motion, a high-performance, 

{'pipeline': 'Insufficient Context Test',
 'latency_seconds': 3.138,
 'answer_tokens': 129,
 'answer': "Apple announced the shipping of Motion, a high-performance, real-time motion graphics design software that integrates with Final Cut Pro HD and DVD Studio Pro 3. It is offered at a breakthrough price of $299. This announcement highlights Apple's expansion in professional video editing tools, bundling Motion with other editing applications [Source 1][Source 2].\n\n- Motion software is designed for motion graphics and integrates with Apple's video editing suite.\n- The price point for Motion is set at $299.\n- This announcement is part of Apple's broader strategy to enhance its professional video editing offerings.\n\nThere is no information available about any other announcements Apple might have made yesterday."}

# Key takeaway

- Prompt engineering controls how the answer is expressed.
- Context optimization controls how much retrieved evidence reaches the model.
- Query transformation improves the chance of retrieving the right evidence.
- Step-back prompting contributes broader conceptual context.
- Running every transformation can improve recall, but it also increases latency and model calls.

Always evaluate quality, latency, and token usage rather than assuming that more retrieval steps are automatically better.